# 34. OpenCV Computer Vision — `opencv-python`
**Python Libraries Lab · Phase 2 · Computer Vision**

_Read, transform and analyse images and video._

✅ **Runs in Colab** as-is.

OpenCV is the most widely used computer vision library. An image, to OpenCV, is just a NumPy array of pixels, so you can resize it, change its colours, blur it, find its edges, detect shapes and faces, and process video frame by frame. It is fast because the core is written in C plus plus.

**In this notebook:** You will draw a product photo mock up from scratch, convert colours, blur and find edges, count objects with contours, detect a face with a built in model, and process a short series of frames like a video.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q "opencv-python<5"
import importlib.metadata as md
for pkg in ['opencv-python']:
    print(pkg, md.version(pkg))

## Step 1 · An image is an array
Create a blank image and draw shapes on it. Note colours are (Blue, Green, Red).

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
print("OpenCV", cv2.__version__)

def show(*imgs, titles=None, cmap=None):
    fig, axes = plt.subplots(1, len(imgs), figsize=(4 * len(imgs), 4))
    for i, (ax, im) in enumerate(zip(np.atleast_1d(axes), imgs)):
        ax.imshow(im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap=cmap or "gray")
        ax.axis("off"); ax.set_title(titles[i] if titles else "")
    plt.show()

img = np.full((300, 400, 3), 245, dtype=np.uint8)
cv2.circle(img, (90, 150), 50, (40, 90, 230), -1)          # red-ish (BGR)
cv2.rectangle(img, (170, 90), (260, 210), (200, 120, 40), -1)
cv2.ellipse(img, (330, 150), (45, 70), 0, 0, 360, (60, 170, 60), -1)
cv2.putText(img, "ShopEasy", (130, 270), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (30, 30, 30), 2)
print("shape:", img.shape, "| pixel at (150, 90):", img[150, 90])
show(img, titles=["Drawn with OpenCV"])

## Step 2 · Colour spaces, blur and edges
Grayscale for analysis, Gaussian blur to remove noise, Canny to find edges.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
noisy = cv2.add(gray, np.random.default_rng(0).integers(0, 40, gray.shape, dtype=np.uint8))
blur = cv2.GaussianBlur(noisy, (7, 7), 0)
edges = cv2.Canny(blur, 50, 150)
show(noisy, blur, edges, titles=["Gray + noise", "Blurred", "Canny edges"])

## Step 3 · Count objects with contours
Threshold to black and white, then find the outline of each shape.

In [ ]:
_, mask = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
big = [c for c in contours if cv2.contourArea(c) > 2000]
out = img.copy()
for c in big:
    x, y, w, h = cv2.boundingRect(c)
    cv2.rectangle(out, (x, y), (x + w, y + h), (0, 0, 255), 2)
print("Objects found:", len(big), "| areas:", [int(cv2.contourArea(c)) for c in big])
show(mask, out, titles=["Mask", "Detected objects"])

## Step 4 · Face detection with a built-in model
A Haar cascade ships with OpenCV. We use a sample photo from scikit-image.

In [ ]:
from skimage import data
photo = cv2.cvtColor(data.astronaut(), cv2.COLOR_RGB2BGR)
face_model = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
faces = face_model.detectMultiScale(cv2.cvtColor(photo, cv2.COLOR_BGR2GRAY), scaleFactor=1.1, minNeighbors=5)
for (x, y, w, h) in faces:
    cv2.rectangle(photo, (x, y), (x + w, y + h), (0, 255, 0), 3)
print("Faces found:", len(faces))
show(photo, titles=["Face detection"])

## Step 5 · Video is just frames
Make a short clip of a moving ball, write it to MP4, read it back frame by frame.

In [ ]:
writer = cv2.VideoWriter("ball.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 20, (320, 200))
for i in range(60):
    frame = np.full((200, 320, 3), 255, np.uint8)
    cv2.circle(frame, (10 + i * 5, 100), 15, (0, 120, 255), -1)
    writer.write(frame)
writer.release()

cap = cv2.VideoCapture("ball.mp4")
frames = []
while True:
    ok, frame = cap.read()
    if not ok:
        break
    frames.append(frame)
cap.release()
print("Frames read:", len(frames))
show(frames[0], frames[30], frames[-1], titles=["first", "middle", "last"])

## ✍️ Your turn
Convert `img` to grayscale and store the **mean pixel brightness** (a float) in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    exp = float(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).mean())
    ok = answer is not None and abs(answer - exp) < 1e-6
    return ok, f"Expected about {exp:.2f}, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = float(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).mean())

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **BGR**
- **Grayscale**
- **Blur**
- **Canny edges**
- **Contours**
- **Haar cascade**

**Next:** 35. Pillow Image Editing (`pillow`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/35_pillow.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install opencv-python<5
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.